# Q5 — SiamFC with Kalman & Particle Filters

## Install & Import

In [1]:
!git clone https://github.com/huanglianghua/siamfc-pytorch.git 2>/dev/null || echo 'already cloned'
!pip install got10k --quiet
!pip install matplotlib pandas scipy --quiet


  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 3.4 MB/s eta 0:00:00


In [2]:
import os
import sys
import glob
from dataclasses import dataclass
from tqdm import tqdm

import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
from scipy.ndimage import label, maximum_filter
from IPython.display import Video, display


In [3]:
REPO_DIR = os.path.abspath("./siamfc-pytorch")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from siamfc.siamfc import TrackerSiamFC
from siamfc import ops


## Helper Functions

In [4]:
def apce(response: np.ndarray) -> float:
    r = response.astype(np.float32)
    r_max = float(r.max())
    r_min = float(r.min())
    denom = np.mean((r - r_min) ** 2) + 1e-12
    return ((r_max - r_min) ** 2) / denom


def peak_ratio_spatial(response: np.ndarray, min_distance: int = 5) -> float:
    r = response.astype(np.float32)
    local_max_mask = (maximum_filter(r, size=min_distance) == r)
    labeled, num_features = label(local_max_mask)
    if num_features < 2:
        return 0.0
    peak_vals = [
        float(r[labeled == i].max())
        for i in range(1, num_features + 1)
    ]
    peak_vals.sort(reverse=True)
    return peak_vals[1] / (peak_vals[0] + 1e-12)


def peak_ratio(response: np.ndarray) -> float:
    flat = np.sort(response.reshape(-1))
    if flat.size < 2:
        return 0.0
    main   = float(flat[-1])
    second = float(flat[-2])
    return second / (main + 1e-12)


def response_confident(response: np.ndarray,
                        peak_ratio_thr: float = 0.75,
                        apce_thr: float = 10.0,
                        fmax_ratio: float = 1.0,
                        beta: float = 0.8) -> bool:
    return (fmax_ratio > beta) and (apce(response) >= apce_thr)


## BBox dataclass

In [5]:
@dataclass
class BBox:
    x: float
    y: float
    w: float
    h: float

    @property
    def cx(self):
        return self.x + self.w / 2.0

    @property
    def cy(self):
        return self.y + self.h / 2.0

    def as_xywh(self):
        return np.array([self.x, self.y, self.w, self.h], dtype=np.float32)

    def as_cxcywh(self):
        return np.array([self.cx, self.cy, self.w, self.h], dtype=np.float32)

    @staticmethod
    def from_cxcywh(cx, cy, w, h):
        return BBox(cx - w / 2.0, cy - h / 2.0, w, h)


## Kalman Filter 

In [6]:
class KalmanTracker:

    def __init__(self):
        self.initialized = False

    def init(self, bbox: BBox):
        self.x = np.array([bbox.cx, bbox.cy, 0.0, 0.0, bbox.w, bbox.h], dtype=np.float32)
        self.P = np.diag([10, 10, 100, 100, 25, 25]).astype(np.float32)
        self.Q = np.diag([1,  1,  5,   5,   2,  2 ]).astype(np.float32)
        self.R = np.diag([5,  5,  10,  10           ]).astype(np.float32)
        self.F = np.array([
            [1, 0, 1, 0, 0, 0],
            [0, 1, 0, 1, 0, 0],
            [0, 0, 1, 0, 0, 0],
            [0, 0, 0, 1, 0, 0],
            [0, 0, 0, 0, 1, 0],
            [0, 0, 0, 0, 0, 1],
        ], dtype=np.float32)
        self.H = np.array([
            [1, 0, 0, 0, 0, 0],
            [0, 1, 0, 0, 0, 0],
            [0, 0, 0, 0, 1, 0],
            [0, 0, 0, 0, 0, 1],
        ], dtype=np.float32)
        self.initialized = True

    def predict(self) -> BBox:
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q
        return BBox.from_cxcywh(*self.x[[0, 1, 4, 5]])

    def update(self, bbox: BBox) -> BBox:
        z = np.array([bbox.cx, bbox.cy, bbox.w, bbox.h], dtype=np.float32)
        y = z - (self.H @ self.x)
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T @ np.linalg.inv(S)
        self.x = self.x + K @ y
        self.P = (np.eye(6, dtype=np.float32) - K @ self.H) @ self.P
        return BBox.from_cxcywh(*self.x[[0, 1, 4, 5]])


## Particle Filter

In [7]:
class ParticleTracker:

    def __init__(self, num_particles: int = 200,
                 process_std=(6.0, 6.0, 2.0, 2.0),
                 measurement_std: float = 20.0):
        self.num_particles   = num_particles
        self.process_std     = np.array(process_std, dtype=np.float32)
        self.measurement_std = float(measurement_std)
        self.initialized     = False

    def init(self, bbox: BBox):
        self.particles = np.tile(
            [bbox.cx, bbox.cy, bbox.w, bbox.h],
            (self.num_particles, 1)
        ).astype(np.float32)
        self.weights = np.full(self.num_particles, 1.0 / self.num_particles, dtype=np.float32)
        self.initialized = True

    def estimate(self) -> np.ndarray:
        return np.average(self.particles, axis=0, weights=self.weights)

    def predict(self):
        noise = np.random.randn(self.num_particles, 4).astype(np.float32) * self.process_std
        self.particles += noise
        self.particles[:, 2] = np.maximum(5.0, self.particles[:, 2])  # min width
        self.particles[:, 3] = np.maximum(5.0, self.particles[:, 3])  # min height

    def update(self, measurement: BBox) -> BBox:
        meas = np.array([measurement.cx, measurement.cy], dtype=np.float32)
        diff  = self.particles[:, :2] - meas[None, :]
        dist2 = np.sum(diff ** 2, axis=1)
        likelihood    = np.exp(-0.5 * dist2 / (self.measurement_std ** 2))
        self.weights  = likelihood + 1e-12
        self.weights /= self.weights.sum()

        idx = np.random.choice(self.num_particles, self.num_particles, p=self.weights)
        self.particles = self.particles[idx]
        self.weights.fill(1.0 / self.num_particles)

        est = np.average(self.particles, axis=0, weights=self.weights)
        return BBox.from_cxcywh(est[0], est[1], est[2], est[3])


## Siam with filters

In [8]:
class SiamFCWithFilters:

    def __init__(self,
                 weight_path: str = "/kaggle/input/datasets/sinatb/dl4cv-hw5-q5/pretrained/siamfc_alexnet_e50.pth",
                 mode: str = "baseline",
                 beta: float = 0.8,
                 fmax_window: int = 20):
        self.tracker = TrackerSiamFC(net_path=weight_path)
        self.mode = mode.lower()
        
        self.kf = KalmanTracker()   if self.mode == "kalman"   else None
        self.pf = ParticleTracker() if self.mode == "particle" else None

        self.beta         = beta          
        self.fmax_window  = fmax_window   
        self._fmax_history: list = []     

    def _update_fmax_history(self, response: np.ndarray) -> float:
        f_maxn = float(response.max())
        self._fmax_history.append(f_maxn)
        if len(self._fmax_history) > self.fmax_window:
            self._fmax_history.pop(0)
        if len(self._fmax_history) < 2:
            return 1.0   
        f_maxave = float(np.mean(self._fmax_history[:-1]))  
        return f_maxn / (f_maxave + 1e-12)

    def _update_siamfc_state(self, bbox: BBox):
        t = self.tracker
        t.center = np.array([
            bbox.y + (bbox.h - 1) / 2.0,   # row
            bbox.x + (bbox.w - 1) / 2.0,   # col
        ], dtype=np.float32)
        t.target_sz = np.array([bbox.h, bbox.w], dtype=np.float32)
        context = getattr(t.cfg, 'context', 0.5) * np.sum(t.target_sz)
        t.z_sz  = float(np.sqrt(np.prod(t.target_sz + context)))
        t.x_sz  = t.z_sz * t.cfg.instance_sz / t.cfg.exemplar_sz

    def _extract_peak_candidates(self,
                                  response: np.ndarray,
                                  scale_id: int,
                                  min_distance: int = 5) -> list:
        t   = self.tracker
        r   = response.astype(np.float32)
        local_max_mask = (maximum_filter(r, size=min_distance) == r)
        labeled, num_features = label(local_max_mask)

        candidates = []
        for comp_id in range(1, num_features + 1):
            comp_pixels = np.argwhere(labeled == comp_id)  
            best_idx = np.argmax(r[comp_pixels[:, 0], comp_pixels[:, 1]])
            loc      = comp_pixels[best_idx]               # [row, col]
            peak_val = float(r[loc[0], loc[1]])

            disp_in_response = np.array(loc, dtype=np.float32) - (t.upscale_sz - 1) / 2
            disp_in_instance = disp_in_response * t.cfg.total_stride / t.cfg.response_up
            disp_in_image    = (disp_in_instance * t.x_sz
                                * t.scale_factors[scale_id] / t.cfg.instance_sz)

            # centre in image coordinates for this candidate peak
            cand_center = t.center + disp_in_image
            cx = float(cand_center[1])          # col → x
            cy = float(cand_center[0])          # row → y
            w  = float(t.target_sz[1])
            h  = float(t.target_sz[0])
            candidates.append((peak_val, BBox.from_cxcywh(cx, cy, w, h)))

        candidates.sort(key=lambda x: x[0], reverse=True)
        return [bbox for _, bbox in candidates]

    def _closest_to_prediction(self,
                                candidates: list,
                                prediction: BBox) -> BBox:
        if not candidates:
            return prediction
        px, py = prediction.cx, prediction.cy
        return min(candidates,
                   key=lambda b: (b.cx - px) ** 2 + (b.cy - py) ** 2)

    def _siamfc_measure(self, frame_bgr: np.ndarray):
        t = self.tracker
        t.net.eval()

        x = [
            ops.crop_and_resize(
                frame_bgr, t.center, t.x_sz * f,
                out_size=t.cfg.instance_sz,
                border_value=t.avg_color,
            )
            for f in t.scale_factors
        ]
        x = np.stack(x, axis=0)
        x = torch.from_numpy(x).to(t.device).permute(0, 3, 1, 2).float()

        x         = t.net.backbone(x)
        responses = t.net.head(t.kernel, x).squeeze(1).cpu().detach().numpy()

        responses = np.stack([
            cv2.resize(u, (t.upscale_sz, t.upscale_sz), interpolation=cv2.INTER_CUBIC)
            for u in responses
        ])
        responses[:t.cfg.scale_num // 2]     *= t.cfg.scale_penalty
        responses[t.cfg.scale_num // 2 + 1:] *= t.cfg.scale_penalty

        scale_id  = np.argmax(np.amax(responses, axis=(1, 2)))
        response  = responses[scale_id]
        response -= response.min()
        response /= response.sum() + 1e-16
        response  = ((1 - t.cfg.window_influence) * response
                     + t.cfg.window_influence * t.hann_window)

        loc              = np.unravel_index(response.argmax(), response.shape)
        disp_in_response = np.array(loc) - (t.upscale_sz - 1) / 2
        disp_in_instance = disp_in_response * t.cfg.total_stride / t.cfg.response_up
        disp_in_image    = (disp_in_instance * t.x_sz
                            * t.scale_factors[scale_id] / t.cfg.instance_sz)

        t.center    += disp_in_image
        scale        = ((1 - t.cfg.scale_lr) * 1.0
                        + t.cfg.scale_lr * t.scale_factors[scale_id])
        t.target_sz *= scale
        t.z_sz      *= scale
        t.x_sz      *= scale

        box = np.array([
            t.center[1] + 1 - (t.target_sz[1] - 1) / 2,
            t.center[0] + 1 - (t.target_sz[0] - 1) / 2,
            t.target_sz[1],
            t.target_sz[0],
        ], dtype=np.float32)

        conf = {
            "apce":       apce(response),
            "peak_ratio": peak_ratio_spatial(response),   
            "response":   response,
            "scale_id":   int(scale_id),
        }
        return box, conf

    def step(self, frame_bgr: np.ndarray,
             peak_ratio_thr: float = 0.75,
             apce_thr: float = 10.0):

        siam_box_xywh, conf = self._siamfc_measure(frame_bgr)
        meas      = BBox(*map(float, siam_box_xywh))

        fmax_ratio = self._update_fmax_history(conf["response"])

        confident = response_confident(
            conf["response"],
            peak_ratio_thr=peak_ratio_thr,
            apce_thr=apce_thr,
            fmax_ratio=fmax_ratio,
            beta=self.beta,
        )

        if self.mode == "baseline":
            final_box = meas

        elif self.mode == "kalman":
            pred = self.kf.predict()    

            if confident:
                spr = conf["peak_ratio"]   
                if spr > peak_ratio_thr:
                    candidates = self._extract_peak_candidates(
                        conf["response"], conf["scale_id"])
                    meas = self._closest_to_prediction(candidates, pred)
                final_box = self.kf.update(meas)
            else:
                final_box = pred        

            self._update_siamfc_state(final_box)

        else: 
            self.pf.predict()           
            est_bbox = BBox.from_cxcywh(*self.pf.estimate())

            if confident:
                spr = conf["peak_ratio"]
                if spr > peak_ratio_thr:
                    candidates = self._extract_peak_candidates(
                        conf["response"], conf["scale_id"])
                    meas = self._closest_to_prediction(candidates, est_bbox)
                final_box = self.pf.update(meas)
            else:
                final_box = est_bbox

            self._update_siamfc_state(final_box)

        self.prev_bbox = final_box
        return final_box.as_xywh(), conf, confident


    def init(self, frame_bgr: np.ndarray, init_bbox_xywh):
        x, y, w, h = map(float, init_bbox_xywh)
        self.prev_bbox = BBox(x, y, w, h)
        self.tracker.init(frame_bgr, np.array([x, y, w, h], dtype=np.float32))
        if self.kf: self.kf.init(self.prev_bbox)
        if self.pf: self.pf.init(self.prev_bbox)
        self._fmax_history = []   # reset history on re-init


## Load Data

In [10]:
FRAMES_DIR = "/kaggle/input/datasets/sinatb/dl4cv-hw5-q5/data/godfather_color"
GT_FILE    = "/kaggle/input/datasets/sinatb/dl4cv-hw5-q5/data/godfather_anno/groundtruth_rect.txt"
WEIGHTS    = "/kaggle/input/datasets/sinatb/dl4cv-hw5-q5/pretrained/siamfc_alexnet_e50.pth"

frames = sorted(glob.glob(os.path.join(FRAMES_DIR, "*.jpg")))
print(f"Frames : {len(frames)}")

gt_boxes = []
with open(GT_FILE) as f:
    for line in f:
        vals = [float(v) for v in line.replace(",", " ").split()]
        gt_boxes.append(vals[:4])
gt_boxes = np.array(gt_boxes)
print(f"GT boxes: {len(gt_boxes)}")
assert len(frames) == len(gt_boxes), "Frame/annotation count mismatch!"


Frames : 366
GT boxes: 366


In [11]:
def iou(box1, box2) -> float:
    x1, y1, w1, h1 = box1
    x2, y2, w2, h2 = box2

    xa = max(x1, x2)
    ya = max(y1, y2)
    xb = min(x1 + w1, x2 + w2)
    yb = min(y1 + h1, y2 + h2)   
    
    inter = max(0.0, xb - xa) * max(0.0, yb - ya)
    union = w1 * h1 + w2 * h2 - inter
    return float(inter / (union + 1e-8))


In [12]:
def evaluate_run(preds: np.ndarray, gt: np.ndarray) -> dict:
    ious = np.array([iou(p, g) for p, g in zip(preds, gt)])
    return {
        "mean_iou":   float(np.mean(ious)),
        "success50":  float(np.mean(ious > 0.5)),
        "success75":  float(np.mean(ious > 0.75)),
        "ious":        ious,
    }


In [13]:
def run_sequence(mode: str = "baseline",
                 peak_thr: float = 0.75,
                 apce_thr: float = 10.0,
                 beta: float = 0.8) -> tuple:
    model = SiamFCWithFilters(weight_path=WEIGHTS, mode=mode, beta=beta)
    first = cv2.imread(frames[0])
    model.init(first, gt_boxes[0])

    preds        = [np.array(gt_boxes[0], dtype=np.float32)]
    apce_scores  = [100.0]   
    conf_flags   = [True]

    for fp in tqdm(frames[1:], desc=f"{mode:8s} apce≥{apce_thr:.0f}", leave=False):
        frame = cv2.imread(fp)
        bbox, conf_dict, ok = model.step(frame,
                                          peak_ratio_thr=peak_thr,
                                          apce_thr=apce_thr)
        preds.append(np.array(bbox, dtype=np.float32))
        apce_scores.append(float(conf_dict["apce"]))
        conf_flags.append(bool(ok))

    return np.stack(preds), np.array(apce_scores), np.array(conf_flags)


## Experiments

In [16]:
PEAK_THR   = 0.75   
APCE_PAPER = 10.0   

EXPERIMENTS = [
    ("baseline",     "baseline", APCE_PAPER),       
    ("kalman_A0",    "kalman",   APCE_PAPER - 10),  
    ("kalman_A10",   "kalman",   APCE_PAPER),       
    ("kalman_A20",   "kalman",   APCE_PAPER + 10),  
    ("particle_A0",  "particle", APCE_PAPER - 10),  
    ("particle_A10", "particle", APCE_PAPER),       
    ("particle_A20", "particle", APCE_PAPER + 10),  
]

print(f"Running {len(EXPERIMENTS)} experiments ...\n")
results = {}
for name, mode, apce_thr in EXPERIMENTS:
    preds, apce_sc, flags = run_sequence(mode=mode, peak_thr=PEAK_THR, apce_thr=apce_thr)
    results[name] = (preds, apce_sc, flags)
    metrics = evaluate_run(preds, gt_boxes)
    conf_pct = 100 * flags.mean()
    print(f"  {name:<14s}  mean_IoU={metrics['mean_iou']:.4f}"
          f"  S@0.5={metrics['success50']:.3f}"
          f"  S@0.75={metrics['success75']:.3f}"
          f"  conf={conf_pct:.1f}%")


Running 7 experiments ...



  baseline        mean_IoU=0.3496  S@0.5=0.451  S@0.75=0.142  conf=0.3%


  kalman_A0       mean_IoU=0.3570  S@0.5=0.475  S@0.75=0.142  conf=96.2%


  kalman_A10      mean_IoU=0.0260  S@0.5=0.025  S@0.75=0.011  conf=0.8%


  kalman_A20      mean_IoU=0.0260  S@0.5=0.025  S@0.75=0.011  conf=0.3%


  particle_A0     mean_IoU=0.3466  S@0.5=0.295  S@0.75=0.011  conf=98.1%


  particle_A10    mean_IoU=0.0246  S@0.5=0.022  S@0.75=0.008  conf=0.3%


  particle_A20    mean_IoU=0.0262  S@0.5=0.025  S@0.75=0.011  conf=0.3%


## Summary Table

In [17]:
rows = []
for name, mode, apce_thr in EXPERIMENTS:
    preds, apce_sc, flags = results[name]
    m = evaluate_run(preds, gt_boxes)
    rows.append({
        "Experiment":       name,
        "Mode":             mode,
        "APCE thr":         apce_thr,
        "Mean IoU":         round(m["mean_iou"],  4),
        "Success@0.5":      round(m["success50"], 4),
        "Success@0.75":     round(m["success75"], 4),
        "Confident %":      round(100 * flags.mean(), 1),
    })

df = pd.DataFrame(rows)
df_sorted = df.sort_values("Mean IoU", ascending=False).reset_index(drop=True)
df_sorted


,Experiment,Mode,APCE thr,Mean IoU,Success@0.5,Success@0.75,Confident %
0,kalman_A0,kalman,0.0,0.3570,0.4754,0.1421,96.2
1,baseline,baseline,10.0,0.3496,0.4508,0.1421,0.3
2,particle_A0,particle,0.0,0.3466,0.2951,0.0109,98.1
3,particle_A20,particle,20.0,0.0262,0.0246,0.0109,0.3
4,kalman_A10,kalman,10.0,0.0260,0.0246,0.0109,0.8
5,kalman_A20,kalman,20.0,0.0260,0.0246,0.0109,0.3
6,particle_A10,particle,10.0,0.0246,0.0219,0.0082,0.3


## Visualization

In [18]:
CV_COLOR = {
    "baseline": (0,  165, 255),   # orange
    "kalman":   (255, 80,   0),   # blue
    "particle": (0,    0, 220),   # red
}
MPL_COLOR = {
    "baseline": "#FFA500",
    "kalman":   "#0050FF",
    "particle": "#DD0000",
}


def annotate_frame(frame_bgr: np.ndarray,
                   gt_box, pred_box,
                   mode: str, confident: bool,
                   frame_idx: int,
                   iou_val: float,
                   apce_val: float) -> np.ndarray:
    out = frame_bgr.copy()

    gx, gy, gw, gh = (int(v) for v in gt_box)
    cv2.rectangle(out, (gx, gy), (gx + gw, gy + gh), (0, 200, 0), 3)
    cv2.putText(out, 'GT', (gx, max(0, gy - 6)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 200, 0), 2)

    px, py, pw, ph = (int(v) for v in pred_box)
    color     = CV_COLOR.get(mode, (200, 200, 0))
    thickness = 3 if confident else 1
    cv2.rectangle(out, (px, py), (px + pw, py + ph), color, thickness)
    label = f"{mode}" + ("" if confident else " (?)")
    cv2.putText(out, label, (px, max(0, py - 6)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

    overlay = out.copy()
    cv2.rectangle(overlay, (0, 0), (270, 100), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.50, out, 0.50, 0, out)
    cv2.putText(out, f'Frame : {frame_idx:4d}', (8, 22),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)
    iou_c = (0, 230, 120) if iou_val > 0.5 else (0, 60, 255)
    cv2.putText(out, f'IoU   : {iou_val:.3f}', (8, 44), cv2.FONT_HERSHEY_SIMPLEX, 0.55, iou_c, 1)
    cv2.putText(out, f'APCE  : {apce_val:.1f}',  (8, 66), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255,255,255), 1)
    conf_c = (0, 230, 0) if confident else (0, 60, 255)
    cv2.putText(out, 'CONFIDENT' if confident else 'UNCERTAIN', (8, 90),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, conf_c, 2)
    return out


def save_tracking_video(frames_list, gt_arr, pred_arr,
                         apce_arr, flag_arr,
                         mode: str, output_path: str, fps: int = 15):
    """Write an annotated MP4 for one experiment."""
    first = cv2.imread(frames_list[0])
    h, w  = first.shape[:2]
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(output_path, fourcc, fps, (w, h))

    for i, fp in enumerate(frames_list):
        frame     = cv2.imread(fp)
        iou_val   = iou(pred_arr[i], gt_arr[i])
        annotated = annotate_frame(
            frame, gt_arr[i], pred_arr[i],
            mode=mode, confident=bool(flag_arr[i]),
            frame_idx=i, iou_val=iou_val, apce_val=float(apce_arr[i])
        )
        writer.write(annotated)

    writer.release()
    print(f"  saved → {output_path}")


In [19]:
OUT_DIR = "./Q5_out"
os.makedirs(f"{OUT_DIR}/videos", exist_ok=True)

for name, mode, _ in EXPERIMENTS:
    preds, apce_sc, flags = results[name]
    save_tracking_video(
        frames_list  = frames,
        gt_arr       = gt_boxes,
        pred_arr     = preds,
        apce_arr     = apce_sc,
        flag_arr     = flags,
        mode         = mode,
        output_path  = f"{OUT_DIR}/videos/{name}.mp4",
    )


  saved → ./Q5_out/videos/baseline.mp4
  saved → ./Q5_out/videos/kalman_A0.mp4
  saved → ./Q5_out/videos/kalman_A10.mp4
  saved → ./Q5_out/videos/kalman_A20.mp4
  saved → ./Q5_out/videos/particle_A0.mp4
  saved → ./Q5_out/videos/particle_A10.mp4
  saved → ./Q5_out/videos/particle_A20.mp4


# Analysis
Having a small APCE seems to work best in this dataset.
The videos for this question are available [here](https://drive.iust.ac.ir/index.php/s/toaoMRgeWTrR9GH).